In [1]:
from utils import *

import torch as th
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.nn.functional as F

import importlib
import data_handler

importlib.reload(data_handler)

import tqdm


In [38]:
import torch
import torch.nn as nn
import torchvision.models as models

class RNet(nn.Module):
    def __init__(self, num_classes=2):
        super(RNet, self).__init__()
        self.resnet = models.resnet18(pretrained=False)
        
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)

# Example usage:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RNet(num_classes=2).to(device)


In [ ]:
import torch
import torch.nn.utils.prune as prune

def apply_pruning(model, sparsity=0.5):
    """
    Applies unstructured pruning to each layer in the model progressively.
    """
    for name, module in model.named_modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
            prune.l1_unstructured(module, name='weight', amount=sparsity)

def train_and_prune(model, epochs, initial_sparsity=0.0, final_sparsity=0.9):
    """
    Gradually prunes the model over several epochs.
    """
    sparsity_step = (final_sparsity - initial_sparsity) / epochs
    loss_func = nn.CrossEntropyLoss()
    optimizer = th.optim.Adam(model.parameters(), lr=0.0001)

    for epoch in range(epochs):
        current_sparsity = initial_sparsity + epoch * sparsity_step

        # Training code 
        model.train()
        for images, labels in data_handler.tr_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            apply_pruning(model, current_sparsity)
            outputs = model(images)
            loss = loss_func(outputs, labels)
            apply_pruning(model,current_sparsity)
            loss.backward()
            optimizer.step()
        

        # Print progress and validate
        print(f"Epoch {epoch+1}/{epochs}, Sparsity: {current_sparsity:.2f}, Loss: {loss.item()}")

    # Remove pruning re-parametrization to finalize the model's sparsity
    for module in model.modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
            prune.remove(module, 'weight')

    # Fine-tune the pruned model
    fine_tune_epochs = 5  
    for epoch in range(fine_tune_epochs):
        model.train()
        for images, labels in data_handler.tr_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_func(outputs, labels)
            loss.backward()
            optimizer.step()
        print(f"Fine-tuning Epoch {epoch+1}/{fine_tune_epochs}, Loss: {loss.item()}")
    
    torch.save(model.state_dict(), "resnet18_00_90.pt")


In [40]:
train_and_prune(model, 10)

Epoch 1/10, Sparsity: 0.00, Loss: 0.0016448288224637508
Epoch 2/10, Sparsity: 0.09, Loss: 0.680039644241333
Epoch 3/10, Sparsity: 0.18, Loss: 0.7056616544723511
Epoch 4/10, Sparsity: 0.27, Loss: 0.6932121515274048
Epoch 5/10, Sparsity: 0.36, Loss: 0.7031814455986023
Epoch 6/10, Sparsity: 0.45, Loss: 0.6931880712509155
Epoch 7/10, Sparsity: 0.54, Loss: 0.693180501461029
Epoch 8/10, Sparsity: 0.63, Loss: 0.693177342414856
Epoch 9/10, Sparsity: 0.72, Loss: 0.686457097530365
Epoch 10/10, Sparsity: 0.81, Loss: 0.6931692361831665
Fine-tuning Epoch 1/5, Loss: 0.6931636333465576
Fine-tuning Epoch 2/5, Loss: 0.698265016078949
Fine-tuning Epoch 3/5, Loss: 0.6931558847427368
Fine-tuning Epoch 4/5, Loss: 0.6931528449058533
Fine-tuning Epoch 5/5, Loss: 0.6931511759757996


In [41]:
# Load the model
model = RNet().to(device)
#model.load_state_dict(th.load("resnet18_not_pretrained.pt"))
model.load_state_dict(th.load("resnet18_00_90.pt"))

# Test the model
model.eval()

# Test loop
total_images = 0
nr_acc = 0
for images, label in data_handler.te_loader:
    images = images.to(device)
    labels = label.to(device)
    
    # Forward pass
    outputs = model(images)
    
    # Predicted classes
    predicted = th.argmax(outputs, dim=1)
    
    # Accuracy calculation (vectorized)
    nr_acc += (predicted == labels).sum().item()  # Count correct predictions
    total_images += labels.size(0)  # Keep track of the total number of images
    
    
    # Original image
    # plt.imshow(images[0].cpu().permute(1, 2, 0).numpy())
    # plt.title(f"Pred: {"1" if predicted[0] else "0"} | Label: {labels[0]}")
    # plt.axis('off')
    
    # plt.show()

print(nr_acc/total_images)

C:\Users\Danie\AppData\Local\Temp\ipykernel_13784\792102563.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(th.load("resnet18_00_90.pt"))


0.43661971830985913
